# Decision Tree: Complete Guide
### Regression and Classification — Parameters and Hyperparameters

---

**Contents**
1. Theory and How Decision Trees Work
2. All Parameters — Explained
3. Decision Tree Classifier
4. Decision Tree Regressor
5. Hyperparameter Tuning (GridSearchCV)
6. Visualization and Interpretation
7. Overfitting, Pruning, and Best Practices
8. Summary Table

---
## 1. Theory — How Decision Trees Work

A Decision Tree recursively partitions the feature space by selecting the best split at each node using an impurity criterion. The goal is to create pure (homogeneous) leaf nodes.

**Key Concepts:**

| Concept | Description |
|---|---|
| Root Node | The top node representing the entire dataset |
| Splitting | Dividing a node into two child nodes based on a feature threshold |
| Leaf Node | Terminal node that outputs a prediction |
| Depth | Number of levels from root to the deepest leaf |
| Pruning | Removing branches to reduce overfitting |

**Split Criteria for Classification:**
- **Gini Impurity**: $Gini = 1 - \sum_{i=1}^{C} p_i^2$
- **Entropy (Information Gain)**: $H = -\sum_{i=1}^{C} p_i \log_2(p_i)$
- **Log Loss**: Uses log-probability, better for probability calibration

**Split Criteria for Regression:**
- **Squared Error (MSE)**: $MSE = \frac{1}{n}\sum(y_i - \bar{y})^2$ — minimizes variance
- **Friedman MSE**: MSE with Friedman improvement score for better splits
- **Mean Absolute Error (MAE)**: $MAE = \frac{1}{n}\sum|y_i - median(y)|$ — robust to outliers
- **Poisson**: Uses Poisson deviance, suited for count data

**Prediction:**
- Classification: Majority class in leaf
- Regression: Mean (or median for MAE) of target values in leaf

---
## 2. All Parameters — Explained

Below is a complete reference for every parameter in `DecisionTreeClassifier` and `DecisionTreeRegressor`.

### Shared Parameters (Classifier and Regressor)

| Parameter | Type | Default | Description |
|---|---|---|---|
| `criterion` | str | 'gini' / 'squared_error' | Split quality measure. Classifier: 'gini', 'entropy', 'log_loss'. Regressor: 'squared_error', 'friedman_mse', 'absolute_error', 'poisson' |
| `splitter` | str | 'best' | Strategy to choose split at each node. 'best': best split. 'random': best random split (faster, acts as regularization) |
| `max_depth` | int or None | None | Maximum depth of the tree. None means nodes are expanded until all leaves are pure or contain fewer than `min_samples_split` samples. Controls overfitting |
| `min_samples_split` | int or float | 2 | Minimum number of samples required to split an internal node. Float = fraction of total samples |
| `min_samples_leaf` | int or float | 1 | Minimum number of samples required to be at a leaf node. Float = fraction. Acts as smoothing |
| `min_weight_fraction_leaf` | float | 0.0 | Minimum weighted fraction of sum of weights required at a leaf. Useful when `sample_weight` is provided |
| `max_features` | int, float, str, None | None | Number of features to consider at each split. 'sqrt': sqrt(n_features). 'log2': log2(n_features). int: exact count. float: fraction. None: all features |
| `random_state` | int or None | None | Seed for reproducibility. Controls randomness in splitter and feature selection |
| `max_leaf_nodes` | int or None | None | Grow tree with at most this many leaf nodes. None: unlimited. Reduces overfitting |
| `min_impurity_decrease` | float | 0.0 | A node will be split if the impurity decrease is greater than or equal to this value. Acts as pre-pruning |
| `class_weight` | dict, list, 'balanced', None | None | *Classifier only.* Weights for classes. 'balanced': inversely proportional to class frequencies |
| `ccp_alpha` | float | 0.0 | Complexity parameter for Minimal Cost-Complexity Pruning. Larger values prune more |
| `monotonic_cst` | array-like or None | None | Monotonic constraints per feature. 1: increasing, -1: decreasing, 0: none. (sklearn >= 1.4) |

In [ ]:
# Install / upgrade scikit-learn if needed
# !pip install scikit-learn --upgrade

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree,
    export_text
)
from sklearn.datasets import (
    load_iris,
    load_breast_cancer,
    load_diabetes,
    make_regression,
    make_classification
)
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score,
    ConfusionMatrixDisplay
)
from sklearn.inspection import DecisionBoundaryDisplay

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"numpy version:        {np.__version__}")
print(f"pandas version:       {pd.__version__}")

---
## 3. Decision Tree Classifier

### 3.1 Load and Explore Dataset

In [ ]:
# Using Iris (3-class) for interpretability and Breast Cancer (binary) for medical context

# --- Iris ---
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = pd.Series(iris.target, name='species')

print("Iris Dataset")
print(f"  Shape: {X_iris.shape}")
print(f"  Classes: {iris.target_names.tolist()}")
print(f"  Class distribution:\n{y_iris.value_counts()}\n")

# --- Breast Cancer ---
cancer = load_breast_cancer()
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_cancer = pd.Series(cancer.target, name='diagnosis')

print("Breast Cancer Dataset")
print(f"  Shape: {X_cancer.shape}")
print(f"  Classes: {cancer.target_names.tolist()}")
print(f"  Class distribution:\n{y_cancer.value_counts()}")

### 3.2 Criterion Comparison — Gini vs Entropy vs Log Loss

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

criteria = ['gini', 'entropy', 'log_loss']
results = []

for crit in criteria:
    clf = DecisionTreeClassifier(criterion=crit, random_state=42)
    clf.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test))
    cv_score  = cross_val_score(clf, X_iris, y_iris, cv=5).mean()
    results.append({
        'criterion': crit,
        'train_accuracy': round(train_acc, 4),
        'test_accuracy':  round(test_acc,  4),
        'cv_accuracy':    round(cv_score,  4),
        'n_leaves':       clf.get_n_leaves(),
        'depth':          clf.get_depth()
    })

df_criteria = pd.DataFrame(results).set_index('criterion')
print("Criterion Comparison (Iris Dataset)")
print(df_criteria.to_string())

### 3.3 Effect of max_depth — Bias-Variance Tradeoff

In [ ]:
depths = range(1, 15)
train_scores, test_scores, cv_scores = [], [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  clf.predict(X_test)))
    cv_scores.append(cross_val_score(clf, X_iris, y_iris, cv=5).mean())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_scores, marker='o', label='Train Accuracy',  linewidth=2)
ax.plot(depths, test_scores,  marker='s', label='Test Accuracy',   linewidth=2)
ax.plot(depths, cv_scores,    marker='^', label='CV Accuracy (5-fold)', linewidth=2, linestyle='--')
ax.axvline(x=cv_scores.index(max(cv_scores)) + 1, color='gray', linestyle=':', label='Best CV Depth')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of max_depth on Classifier Performance (Iris)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[cv_scores.index(max(cv_scores))]
print(f"Best depth by CV accuracy: {best_depth} (CV Acc = {max(cv_scores):.4f})")

### 3.4 min_samples_split and min_samples_leaf

In [ ]:
# These parameters act as pre-pruning — they stop splits before they happen

param_grid_pre = {
    'min_samples_split': [2, 5, 10, 20, 30],
    'min_samples_leaf':  [1, 2, 5, 10, 15]
}

results_pre = []
for mss in param_grid_pre['min_samples_split']:
    for msl in param_grid_pre['min_samples_leaf']:
        clf = DecisionTreeClassifier(
            min_samples_split=mss,
            min_samples_leaf=msl,
            random_state=42
        )
        cv = cross_val_score(clf, X_iris, y_iris, cv=5).mean()
        results_pre.append({'min_samples_split': mss, 'min_samples_leaf': msl, 'cv_acc': cv})

df_pre = pd.DataFrame(results_pre)
pivot = df_pre.pivot(index='min_samples_split', columns='min_samples_leaf', values='cv_acc')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto',
               vmin=pivot.values.min(), vmax=pivot.values.max())
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('min_samples_split')
ax.set_title('5-Fold CV Accuracy: min_samples_split vs min_samples_leaf')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

best_row = df_pre.loc[df_pre['cv_acc'].idxmax()]
print(f"Best: min_samples_split={int(best_row['min_samples_split'])}, "
      f"min_samples_leaf={int(best_row['min_samples_leaf'])}, "
      f"CV Acc={best_row['cv_acc']:.4f}")

### 3.5 max_features and splitter

In [ ]:
# max_features: controls how many features are considered at each split
# 'best' splitter: always picks the best split among considered features
# 'random' splitter: introduces randomness — useful to reduce variance

# For breast cancer (30 features) — more meaningful to vary max_features
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

max_feat_options = [None, 'sqrt', 'log2', 0.5, 0.3, 5, 10]
splitter_options = ['best', 'random']

rows = []
for mf in max_feat_options:
    for sp in splitter_options:
        clf = DecisionTreeClassifier(max_features=mf, splitter=sp, random_state=42)
        cv = cross_val_score(clf, X_cancer, y_cancer, cv=5).mean()
        clf.fit(X_tr, y_tr)
        rows.append({
            'max_features': str(mf),
            'splitter': sp,
            'cv_acc': round(cv, 4),
            'test_acc': round(accuracy_score(y_te, clf.predict(X_te)), 4)
        })

df_mf = pd.DataFrame(rows)
print("max_features and splitter comparison (Breast Cancer, 30 features)")
print(df_mf.pivot(index='max_features', columns='splitter', values='cv_acc').to_string())

### 3.6 class_weight — Handling Imbalanced Classes

In [ ]:
# Simulate imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, weights=[0.9, 0.1],
    random_state=42
)
X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

print(f"Class distribution — minority: {y_imb.sum()}, majority: {len(y_imb) - y_imb.sum()}")

configs = [
    ('None (default)',  None),
    ('balanced',        'balanced'),
    ('{0:1, 1:5}',     {0: 1, 1: 5}),
    ('{0:1, 1:9}',     {0: 1, 1: 9}),
]

for label, cw in configs:
    clf = DecisionTreeClassifier(class_weight=cw, random_state=42)
    clf.fit(X_imb_tr, y_imb_tr)
    report = classification_report(y_imb_te, clf.predict(X_imb_te), output_dict=True)
    print(f"\nclass_weight={label}")
    print(f"  Accuracy:          {report['accuracy']:.4f}")
    print(f"  Minority Recall:   {report['1']['recall']:.4f}")
    print(f"  Minority F1-Score: {report['1']['f1-score']:.4f}")

### 3.7 min_impurity_decrease — Pre-Pruning by Impurity Threshold

In [ ]:
# A node is split only if the impurity decrease >= min_impurity_decrease
# Higher values = more aggressive pre-pruning = simpler trees

thresholds = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]

rows = []
for t in thresholds:
    clf = DecisionTreeClassifier(min_impurity_decrease=t, random_state=42)
    clf.fit(X_train, y_train)
    rows.append({
        'min_impurity_decrease': t,
        'depth':          clf.get_depth(),
        'n_leaves':       clf.get_n_leaves(),
        'train_acc':      round(accuracy_score(y_train, clf.predict(X_train)), 4),
        'test_acc':       round(accuracy_score(y_test,  clf.predict(X_test)),  4),
    })

df_mid = pd.DataFrame(rows).set_index('min_impurity_decrease')
print("Effect of min_impurity_decrease (Iris)")
print(df_mid.to_string())

### 3.8 ccp_alpha — Minimal Cost-Complexity Pruning (Post-Pruning)

In [ ]:
# ccp_alpha introduces a penalty for tree complexity
# cost(T) = impurity(T) + alpha * |leaves(T)|
# Use cost_complexity_pruning_path() to find effective alphas

clf_full = DecisionTreeClassifier(random_state=42)
path = clf_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas  = path.ccp_alphas    # array of alpha values
impurities  = path.impurities    # corresponding total impurities

print(f"Number of candidate alpha values: {len(ccp_alphas)}")
print(f"Alpha range: [{ccp_alphas[0]:.6f}, {ccp_alphas[-1]:.6f}]")

# Train a tree for each alpha
clfs = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf.fit(X_train, y_train)
    clfs.append(clf)

train_acc_ccp = [accuracy_score(y_train, c.predict(X_train)) for c in clfs]
test_acc_ccp  = [accuracy_score(y_test,  c.predict(X_test))  for c in clfs]
n_leaves_ccp  = [c.get_n_leaves() for c in clfs]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ccp_alphas, n_leaves_ccp, marker='o', markersize=4)
axes[0].set_xlabel('ccp_alpha')
axes[0].set_ylabel('Number of Leaves')
axes[0].set_title('Tree Size vs ccp_alpha')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ccp_alphas, train_acc_ccp, marker='o', markersize=4, label='Train')
axes[1].plot(ccp_alphas, test_acc_ccp,  marker='s', markersize=4, label='Test')
axes[1].set_xlabel('ccp_alpha')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs ccp_alpha')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cost-Complexity Pruning Path (Iris Classifier)', fontsize=13)
plt.tight_layout()
plt.show()

best_idx   = test_acc_ccp.index(max(test_acc_ccp))
best_alpha = ccp_alphas[best_idx]
print(f"Best ccp_alpha by test accuracy: {best_alpha:.6f} (Test Acc = {max(test_acc_ccp):.4f})")

### 3.9 Classifier — GridSearchCV Full Hyperparameter Tuning

In [ ]:
param_grid_clf = {
    'criterion':          ['gini', 'entropy'],
    'max_depth':          [None, 3, 5, 7, 10],
    'min_samples_split':  [2, 5, 10],
    'min_samples_leaf':   [1, 2, 5],
    'max_features':       [None, 'sqrt', 'log2'],
    'ccp_alpha':          [0.0, 0.001, 0.01]
}

grid_clf = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_clf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)
grid_clf.fit(X_train, y_train)

print("Best Parameters (Classifier — Iris):")
for k, v in grid_clf.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV Accuracy: {grid_clf.best_score_:.4f}")

best_clf = grid_clf.best_estimator_
print(f"Test Accuracy:    {accuracy_score(y_test, best_clf.predict(X_test)):.4f}")
print(f"Tree Depth:       {best_clf.get_depth()}")
print(f"Number of Leaves: {best_clf.get_n_leaves()}")

### 3.10 Classifier — Final Evaluation

In [ ]:
y_pred_clf = best_clf.predict(X_test)

print("Classification Report (Best Classifier — Iris)")
print(classification_report(y_test, y_pred_clf, target_names=iris.target_names))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(
    best_clf, X_test, y_test,
    display_labels=iris.target_names,
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

# Feature Importances
importances = pd.Series(best_clf.feature_importances_, index=iris.feature_names).sort_values()
importances.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importances (Gini)')
axes[1].set_xlabel('Importance')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---
## 4. Decision Tree Regressor

### 4.1 Load and Explore Dataset

In [ ]:
# Using Diabetes dataset (real-world) + a synthetic regression dataset

diabetes = load_diabetes()
X_dia = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_dia = pd.Series(diabetes.target, name='progression')

print("Diabetes Dataset")
print(f"  Shape:  {X_dia.shape}")
print(f"  Target: {y_dia.describe().to_dict()}\n")

X_dia_tr, X_dia_te, y_dia_tr, y_dia_te = train_test_split(
    X_dia, y_dia, test_size=0.2, random_state=42
)

# Synthetic dataset with known noise level
X_syn, y_syn = make_regression(n_samples=500, n_features=10, noise=20, random_state=42)
X_syn_tr, X_syn_te, y_syn_tr, y_syn_te = train_test_split(
    X_syn, y_syn, test_size=0.2, random_state=42
)
print(f"Synthetic Dataset — shape: {X_syn.shape}, target range: [{y_syn.min():.1f}, {y_syn.max():.1f}]")

### 4.2 Criterion Comparison — Regression

In [ ]:
# squared_error: minimizes MSE — standard default
# friedman_mse:  uses Friedman's improvement score — often better in practice
# absolute_error: minimizes MAE — robust to outliers but slower
# poisson:        for count/rate data (targets must be non-negative)

reg_criteria = ['squared_error', 'friedman_mse', 'absolute_error']
# Note: 'poisson' requires non-negative targets; skipped here

reg_results = []
for crit in reg_criteria:
    reg = DecisionTreeRegressor(criterion=crit, random_state=42)
    reg.fit(X_dia_tr, y_dia_tr)
    y_hat = reg.predict(X_dia_te)
    cv_r2 = cross_val_score(reg, X_dia, y_dia, cv=5, scoring='r2').mean()
    reg_results.append({
        'criterion':   crit,
        'train_R2':    round(r2_score(y_dia_tr, reg.predict(X_dia_tr)), 4),
        'test_R2':     round(r2_score(y_dia_te, y_hat), 4),
        'cv_R2':       round(cv_r2, 4),
        'test_RMSE':   round(np.sqrt(mean_squared_error(y_dia_te, y_hat)), 3),
        'test_MAE':    round(mean_absolute_error(y_dia_te, y_hat), 3),
        'depth':       reg.get_depth(),
        'n_leaves':    reg.get_n_leaves()
    })

df_reg_crit = pd.DataFrame(reg_results).set_index('criterion')
print("Regression Criterion Comparison (Diabetes Dataset)")
print(df_reg_crit.to_string())

### 4.3 Effect of max_depth on Regression

In [ ]:
depths_reg = range(1, 20)
train_r2, test_r2, cv_r2_list = [], [], []

for d in depths_reg:
    reg = DecisionTreeRegressor(max_depth=d, random_state=42)
    reg.fit(X_dia_tr, y_dia_tr)
    train_r2.append(r2_score(y_dia_tr, reg.predict(X_dia_tr)))
    test_r2.append(r2_score(y_dia_te,  reg.predict(X_dia_te)))
    cv_r2_list.append(cross_val_score(reg, X_dia, y_dia, cv=5, scoring='r2').mean())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths_reg, train_r2,    marker='o', label='Train R2',       linewidth=2)
ax.plot(depths_reg, test_r2,     marker='s', label='Test R2',        linewidth=2)
ax.plot(depths_reg, cv_r2_list,  marker='^', label='CV R2 (5-fold)', linewidth=2, linestyle='--')
best_d = depths_reg[cv_r2_list.index(max(cv_r2_list))]
ax.axvline(x=best_d, color='gray', linestyle=':', label=f'Best CV Depth={best_d}')
ax.set_xlabel('max_depth')
ax.set_ylabel('R² Score')
ax.set_title('Effect of max_depth on Regressor Performance (Diabetes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best depth by CV R2: {best_d} (CV R2 = {max(cv_r2_list):.4f})")

### 4.4 max_leaf_nodes and min_weight_fraction_leaf

In [ ]:
# max_leaf_nodes: limits tree by leaf count rather than depth
# Grows the best-first tree (most impurity-reducing splits first)

print("Effect of max_leaf_nodes on Regressor (Diabetes)")
print(f"{'max_leaf_nodes':>20} {'n_leaves':>10} {'depth':>8} {'test_R2':>10} {'test_RMSE':>12}")
print("-" * 65)

for mln in [None, 2, 4, 8, 16, 32, 64, 128]:
    reg = DecisionTreeRegressor(max_leaf_nodes=mln, random_state=42)
    reg.fit(X_dia_tr, y_dia_tr)
    y_hat = reg.predict(X_dia_te)
    print(f"{str(mln):>20} {reg.get_n_leaves():>10} {reg.get_depth():>8} "
          f"{r2_score(y_dia_te, y_hat):>10.4f} "
          f"{np.sqrt(mean_squared_error(y_dia_te, y_hat)):>12.3f}")

print()
print("Effect of min_weight_fraction_leaf (uniform sample weights)")
print(f"{'min_weight_frac':>18} {'n_leaves':>10} {'test_R2':>10}")
print("-" * 42)

for mwf in [0.0, 0.01, 0.05, 0.1, 0.2, 0.3]:
    reg = DecisionTreeRegressor(min_weight_fraction_leaf=mwf, random_state=42)
    reg.fit(X_dia_tr, y_dia_tr)
    y_hat = reg.predict(X_dia_te)
    print(f"{mwf:>18} {reg.get_n_leaves():>10} {r2_score(y_dia_te, y_hat):>10.4f}")

### 4.5 ccp_alpha — Pruning for Regression

In [ ]:
reg_full = DecisionTreeRegressor(random_state=42)
path_reg = reg_full.cost_complexity_pruning_path(X_dia_tr, y_dia_tr)
ccp_alphas_reg = path_reg.ccp_alphas

regs, tr_r2, te_r2 = [], [], []
for alpha in ccp_alphas_reg:
    r = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    r.fit(X_dia_tr, y_dia_tr)
    regs.append(r)
    tr_r2.append(r2_score(y_dia_tr, r.predict(X_dia_tr)))
    te_r2.append(r2_score(y_dia_te, r.predict(X_dia_te)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ccp_alphas_reg, [r.get_n_leaves() for r in regs], marker='o', markersize=4)
axes[0].set_xlabel('ccp_alpha')
axes[0].set_ylabel('Number of Leaves')
axes[0].set_title('Tree Size vs ccp_alpha (Regressor)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ccp_alphas_reg, tr_r2, marker='o', markersize=4, label='Train R2')
axes[1].plot(ccp_alphas_reg, te_r2, marker='s', markersize=4, label='Test R2')
axes[1].set_xlabel('ccp_alpha')
axes[1].set_ylabel('R² Score')
axes[1].set_title('R2 vs ccp_alpha (Regressor)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cost-Complexity Pruning — Diabetes Regressor', fontsize=13)
plt.tight_layout()
plt.show()

best_i = te_r2.index(max(te_r2))
print(f"Best ccp_alpha: {ccp_alphas_reg[best_i]:.6f} | Test R2: {max(te_r2):.4f}")

### 4.6 Regressor — GridSearchCV Full Hyperparameter Tuning

In [ ]:
param_grid_reg = {
    'criterion':          ['squared_error', 'friedman_mse', 'absolute_error'],
    'max_depth':          [None, 3, 5, 7, 10],
    'min_samples_split':  [2, 5, 10],
    'min_samples_leaf':   [1, 2, 5],
    'max_features':       [None, 'sqrt', 'log2'],
    'ccp_alpha':          [0.0, 0.1, 1.0]
}

grid_reg = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_reg,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True
)
grid_reg.fit(X_dia_tr, y_dia_tr)

print("Best Parameters (Regressor — Diabetes):")
for k, v in grid_reg.best_params_.items():
    print(f"  {k}: {v}")

best_reg = grid_reg.best_estimator_
y_hat_best = best_reg.predict(X_dia_te)

print(f"\nBest CV R2:   {grid_reg.best_score_:.4f}")
print(f"Test R2:      {r2_score(y_dia_te, y_hat_best):.4f}")
print(f"Test RMSE:    {np.sqrt(mean_squared_error(y_dia_te, y_hat_best)):.3f}")
print(f"Test MAE:     {mean_absolute_error(y_dia_te, y_hat_best):.3f}")

### 4.7 Regressor — Final Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_dia_te, y_hat_best, alpha=0.6, edgecolors='k', linewidth=0.4)
lims = [min(y_dia_te.min(), y_hat_best.min()), max(y_dia_te.max(), y_hat_best.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual (R2 = {r2_score(y_dia_te, y_hat_best):.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_dia_te - y_hat_best
axes[1].scatter(y_hat_best, residuals, alpha=0.6, edgecolors='k', linewidth=0.4)
axes[1].axhline(0, color='r', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Best Decision Tree Regressor — Diabetes Dataset', fontsize=13)
plt.tight_layout()
plt.show()

# Feature importances
fi_reg = pd.Series(best_reg.feature_importances_, index=diabetes.feature_names).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
fi_reg.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances — Best Regressor (Diabetes)')
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 5. Visualization and Interpretation

### 5.1 Plot the Tree Structure

In [ ]:
# Visualize a shallow, interpretable classifier tree on Iris
clf_vis = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
clf_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(
    clf_vis,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax
)
ax.set_title('Decision Tree Classifier (max_depth=3, Iris)', fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 Text Rule Export

In [ ]:
# export_text: readable text representation of the tree rules
rules = export_text(
    clf_vis,
    feature_names=list(iris.feature_names)
)
print("Decision Tree Rules:")
print(rules)

### 5.3 Decision Boundary Visualization (2 Features)

In [ ]:
# Use only 2 features for 2D boundary plot
feature_pairs = [(0, 2), (2, 3)]  # sepal length/petal length, petal length/width
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (f1, f2) in zip(axes, feature_pairs):
    X2 = X_iris.iloc[:, [f1, f2]].values
    clf2 = DecisionTreeClassifier(max_depth=4, random_state=42)
    clf2.fit(X2, y_iris)
    DecisionBoundaryDisplay.from_estimator(
        clf2, X2,
        response_method='predict',
        cmap='RdYlBu',
        alpha=0.4,
        ax=ax
    )
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=y_iris, cmap='RdYlBu',
                         edgecolors='k', linewidth=0.5, s=40)
    ax.set_xlabel(iris.feature_names[f1])
    ax.set_ylabel(iris.feature_names[f2])
    ax.set_title(f'Decision Boundary\n{iris.feature_names[f1]} vs {iris.feature_names[f2]}')

plt.suptitle('Decision Boundaries — Iris Classifier (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.show()

### 5.4 Learning Curves

In [ ]:
def plot_learning_curve(estimator, X, y, title, scoring='accuracy', cv=5):
    train_sizes, train_sc, val_sc = learning_curve(
        estimator, X, y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=cv, scoring=scoring, n_jobs=-1
    )
    train_mean = train_sc.mean(axis=1)
    train_std  = train_sc.std(axis=1)
    val_mean   = val_sc.mean(axis=1)
    val_std    = val_sc.std(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_sizes, train_mean, 'o-', label='Training Score',   color='steelblue')
    ax.plot(train_sizes, val_mean,   's-', label='Validation Score', color='darkorange')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='steelblue')
    ax.fill_between(train_sizes, val_mean   - val_std,   val_mean   + val_std,   alpha=0.15, color='darkorange')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel(scoring)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Overfit tree vs pruned tree
plot_learning_curve(
    DecisionTreeClassifier(random_state=42),
    X_iris, y_iris,
    'Learning Curve — Unpruned Classifier (Iris)',
    scoring='accuracy'
)

plot_learning_curve(
    DecisionTreeClassifier(max_depth=3, ccp_alpha=0.01, random_state=42),
    X_iris, y_iris,
    'Learning Curve — Pruned Classifier max_depth=3, ccp_alpha=0.01 (Iris)',
    scoring='accuracy'
)

---
## 6. Overfitting, Pruning, and Best Practices

### 6.1 Demonstrating Overfitting

In [ ]:
# A fully grown tree memorizes training data
clf_overfit = DecisionTreeClassifier(random_state=42)  # no constraints
clf_overfit.fit(X_train, y_train)

clf_pruned = DecisionTreeClassifier(max_depth=4, min_samples_leaf=3,
                                    ccp_alpha=0.005, random_state=42)
clf_pruned.fit(X_train, y_train)

print("Overfitting vs Pruning Comparison")
print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10} {'Depth':>8} {'Leaves':>8}")
print("-" * 70)

for label, model in [('Unpruned (default)', clf_overfit), ('Pruned', clf_pruned)]:
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test,  model.predict(X_test))
    print(f"{label:<30} {tr:>10.4f} {te:>10.4f} {model.get_depth():>8} {model.get_n_leaves():>8}")

### 6.2 Pre-Pruning vs Post-Pruning Strategies

In [ ]:
strategies = {
    'No Pruning':                     DecisionTreeClassifier(random_state=42),
    'Pre: max_depth=4':               DecisionTreeClassifier(max_depth=4, random_state=42),
    'Pre: min_samples_leaf=5':        DecisionTreeClassifier(min_samples_leaf=5, random_state=42),
    'Pre: min_impurity_decrease=0.01':DecisionTreeClassifier(min_impurity_decrease=0.01, random_state=42),
    'Pre: max_leaf_nodes=10':         DecisionTreeClassifier(max_leaf_nodes=10, random_state=42),
    'Post: ccp_alpha=0.005':          DecisionTreeClassifier(ccp_alpha=0.005, random_state=42),
    'Combined':                       DecisionTreeClassifier(
                                          max_depth=5, min_samples_leaf=3,
                                          ccp_alpha=0.002, random_state=42
                                      )
}

rows = []
for name, model in strategies.items():
    cv = cross_val_score(model, X_iris, y_iris, cv=5).mean()
    model.fit(X_train, y_train)
    rows.append({
        'strategy':   name,
        'cv_acc':     round(cv, 4),
        'test_acc':   round(accuracy_score(y_test, model.predict(X_test)), 4),
        'depth':      model.get_depth(),
        'n_leaves':   model.get_n_leaves()
    })

df_strategies = pd.DataFrame(rows).set_index('strategy')
print("Pruning Strategy Comparison")
print(df_strategies.to_string())

### 6.3 Stability — Decision Trees are High Variance

In [ ]:
# Small changes in training data can produce very different trees
n_runs = 20
acc_unpruned, acc_pruned = [], []

for seed in range(n_runs):
    X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
        X_iris, y_iris, test_size=0.2, random_state=seed
    )
    c1 = DecisionTreeClassifier(random_state=seed).fit(X_tr_s, y_tr_s)
    c2 = DecisionTreeClassifier(max_depth=4, ccp_alpha=0.01, random_state=seed).fit(X_tr_s, y_tr_s)
    acc_unpruned.append(accuracy_score(y_te_s, c1.predict(X_te_s)))
    acc_pruned.append(accuracy_score(y_te_s, c2.predict(X_te_s)))

print("Model Stability Across 20 Random Train/Test Splits")
print(f"Unpruned — Mean: {np.mean(acc_unpruned):.4f}, Std: {np.std(acc_unpruned):.4f}")
print(f"Pruned   — Mean: {np.mean(acc_pruned):.4f},  Std: {np.std(acc_pruned):.4f}")
print("\nHigher variance in unpruned trees confirms overfitting and instability.")
print("Pruning reduces variance at a slight cost to bias.")
print("For production use, consider ensembles (Random Forest, Gradient Boosting).")

---
## 7. Summary Table — All Parameters

| Parameter | Classifier Default | Regressor Default | Effect When Increased | Use Case |
|---|---|---|---|---|
| `criterion` | 'gini' | 'squared_error' | N/A (categorical) | Choose based on data type and task |
| `splitter` | 'best' | 'best' | N/A | 'random' for speed or regularization |
| `max_depth` | None | None | Less overfitting | Most important pruning parameter |
| `min_samples_split` | 2 | 2 | Smaller tree, less overfitting | Noisy data |
| `min_samples_leaf` | 1 | 1 | Smoother predictions | Regression, imbalanced classes |
| `min_weight_fraction_leaf` | 0.0 | 0.0 | Larger min leaf requirement | Weighted samples |
| `max_features` | None | None | Reduces correlation between splits | High-dimensional data |
| `random_state` | None | None | N/A | Always set for reproducibility |
| `max_leaf_nodes` | None | None | Smaller tree (best-first growth) | Alternative to max_depth |
| `min_impurity_decrease` | 0.0 | 0.0 | Fewer splits, simpler tree | When impurity gains are noisy |
| `class_weight` | None | N/A | Penalizes majority class more | Imbalanced classification |
| `ccp_alpha` | 0.0 | 0.0 | More pruning (post-hoc) | Use pruning path to choose |

### Key Decision Rules

| Situation | Recommended Action |
|---|---|
| Train acc >> Test acc | Reduce max_depth, increase min_samples_leaf, add ccp_alpha |
| Both accuracies low | Increase max_depth, decrease min_samples_split/leaf |
| Imbalanced classes | Set class_weight='balanced' or provide custom weights |
| Outliers in target (regression) | Use criterion='absolute_error' |
| Count/rate target | Use criterion='poisson' |
| Need probability outputs | Use criterion='log_loss' for better calibration |
| High variance, unstable | Use ccp_alpha from pruning path, or switch to Random Forest |
| Interpretability required | Set max_depth <= 4, export_text() for rules |

In [ ]:
# Quick-reference: print all parameters for both estimators
import inspect

clf_params = DecisionTreeClassifier().get_params()
reg_params = DecisionTreeRegressor().get_params()

print("DecisionTreeClassifier — Default Parameters")
for k, v in clf_params.items():
    print(f"  {k:<30}: {v}")

print("\nDecisionTreeRegressor — Default Parameters")
for k, v in reg_params.items():
    print(f"  {k:<30}: {v}")